In [26]:
import polars as pl
from pathlib import Path

In [27]:
DATA_GENERAL = Path("../data_general")
OUT_DATA = Path('../output') 

In [28]:
general_df = (
    pl.scan_parquet(DATA_GENERAL / "spotify_audio_features_*.parquet")
    .filter(pl.col('null_response') == 0)   
    .drop('null_response')
    .rename({'id': 'spotify_track_uri'})
)

In [29]:
song_uri = '5GpfyJrKHJI36jDKtQiyGN'

In [30]:
target_song = general_df.filter(pl.col('spotify_track_uri') == song_uri).collect()
if target_song.is_empty():
    raise ValueError(f"Track with URI {song_uri} not found.")

In [31]:
display(target_song)

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""5GpfyJrKHJI36jDKtQiyGN""","""Golden Brown""",19,206760,3,6,1,93.751,0.563,0.383,-15.474,0.0305,0.144,0.116,0.138,0.589


In [32]:
target_stats = target_song.to_dicts()[0]

In [33]:
# Subjectife paramaters (value given in percentage according to reverens value,
#  like 0.1 is mean : from 90% to 110% of reverense value)


smart_filter = [
    (0.0001,500),
    (0.001, 50),
    (0.01, 5),
    (0.1, 0.5),
    (0.2, 0.25),
    (0.4, 0.15),
    (0.6, 0.1),
    (0.8, 0.07),
    (1, 0.05),
]

subj_params = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']

cofs = {}

for param in subj_params:
    val = target_stats[param]
    cofs[param] = next((cof for limit, cof in smart_filter if val < limit), 0.05) # ?

# Technical paramaters (in absalute value)
cof_tempo = 5 #bbm
cof_loud = 2 #Db

max_attempts = 5
multiplier = 1 # Dont touch
real_multiplier = 1.2

In [34]:
print(cofs)

{'danceability': 0.1, 'energy': 0.15, 'speechiness': 0.5, 'acousticness': 0.25, 'instrumentalness': 0.25, 'liveness': 0.25, 'valence': 0.1}


In [40]:
result = pl.DataFrame()
for attempt in range(max_attempts):
    subj_conditions = [
        pl.col(p).is_between(
            target_stats[p] * (1 - cofs[p] * multiplier),
            target_stats[p] * (1 + cofs[p] * multiplier)
        )
        for p in subj_params
    ]

    t = target_stats['tempo']
    tempo_cond = (
        pl.col('tempo').is_between(t - cof_tempo, t + cof_tempo) |
        pl.col('tempo').is_between((t - cof_tempo) / 2, (t + cof_tempo) / 2) |
        pl.col('tempo').is_between((t - cof_tempo) * 2, (t + cof_tempo) * 2)
    )
    candidate_df = (
        general_df
        .filter(subj_conditions)
        .filter(pl.col('loudness').is_between(target_stats['loudness'] - cof_loud, target_stats['loudness'] + cof_loud))
        .filter(
            (pl.col('key') == target_stats['key']) & 
            (pl.col('time_signature') == target_stats['time_signature']) & 
            (pl.col('mode') == target_stats['mode'])
        )
        .filter(tempo_cond)
        .collect()
        .unique(subset=['name'], keep='first')
    )    
    if len(candidate_df) >= 10:
        result = candidate_df
        break
    multiplier *= real_multiplier


In [45]:
if not result.is_empty():
    result = result.with_columns(
        pl.format("spotify:track:{}", pl.col("spotify_track_uri")).alias("spotify_track_uri")
    )
    out_file = OUT_DATA / f"Like_{target_stats['name']}.csv"
    result.select('spotify_track_uri').write_csv(out_file)
    print(len(result))

13
